# Q3: Geometric-SMOTE Region Benchmark

This notebook starts Question 3 from `manuscript/experiment_plan.md`:

> Does MIMIC compete with Geometric SMOTE on region-based oversampling?

Reusable experiment machinery lives in `src/mimic_experiments/q3_geometric_smote.py`. This notebook chooses parameters, calls that module, and displays saved result tables and plots.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

from IPython.display import display

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "mimic").exists())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import GenerationPolicy

from mimic_experiments.q3_geometric_smote import (
    Q3Config,
    load_q3_dataset,
    load_q3_result_tables,
    plot_q3_metric_comparison,
    print_progress_event,
    published_reference_table,
    q3_dataset_registry,
    q3_result_table_manifest,
    q3_result_table_paths,
    run_q3_benchmark,
)


## Experiment Controls

Use `RUN_PROFILE = "run_full"` to execute and save the repeated fivefold benchmark. Use `RUN_PROFILE = "view"` to skip fitting and regenerate displays from saved CSV tables.

In [ ]:
# Dataset and run controls
DATASET = "pima"  # use a registry row number, or one of the ready keys: "pima", "iris", "wine"
RUN_PROFILE = "view"  # "run_full" or "view"
RANDOM_STATE = 0
N_REPEATS = 5
N_SPLITS = 5
N_JOBS = -1
ARTIFACT_DIR = str(PROJECT_ROOT / "manuscript" / "artifacts" / "q3_geometric_smote")
WORKER_THREADS = 1

# MIMIC controls
MIMIC_MODES = ("identity", "factorised")
MIMIC_CAPACITY_RUN_FULL = 0.25
MIMIC_FEATURE_N_JOBS = 1  # keep 1 when outer jobs parallelize work; increase when running one job at a time

# Classifier controls from the Geometric SMOTE paper
CLASSIFIERS = ("GBC", "LR")

# MIMIC generation policy controls
POLICY_METHOD = "smote"
POLICY_NEIGHBOUR_MODE = "normal"
POLICY_N_NEIGHBORS = 5
POLICY_LAMBDA_RANGE = (0.0, 1.0)

registry = q3_dataset_registry()
DATASET_KEY = registry.loc[DATASET, "key"] if isinstance(DATASET, int) else DATASET

config = Q3Config(
    dataset_key=DATASET_KEY,
    run_profile=RUN_PROFILE,
    random_state=RANDOM_STATE,
    n_repeats=N_REPEATS,
    n_splits=N_SPLITS,
    n_jobs=N_JOBS,
    artifact_dir=ARTIFACT_DIR,
    worker_threads=WORKER_THREADS,
    classifiers=CLASSIFIERS,
    mimic_modes=MIMIC_MODES,
    mimic_capacity_run_full=MIMIC_CAPACITY_RUN_FULL,
    mimic_feature_n_jobs=MIMIC_FEATURE_N_JOBS,
    policy=GenerationPolicy(
        method=POLICY_METHOD,
        neighbour_mode=POLICY_NEIGHBOUR_MODE,
        n_neighbors=POLICY_N_NEIGHBORS,
        lambda_range=POLICY_LAMBDA_RANGE,
    ),
)
config


## Dataset Registry

The registry records the dataset distributions reported by the Geometric SMOTE paper and which exact loaders are currently available.

In [ ]:
display(
    q3_dataset_registry(
        artifact_dir=ARTIFACT_DIR,
        run_profile=RUN_PROFILE,
        mimic_modes=MIMIC_MODES,
        mimic_capacity=config.mimic_capacity,
        policy=config.policy,
        random_state=RANDOM_STATE,
    )
)


## Published Reference Table

In [ ]:
display(published_reference_table())


## Load Dataset

In [ ]:
if config.should_run_experiment:
    df = load_q3_dataset(config)
    display(df.head())
    display(df["label"].value_counts().rename_axis("label").to_frame("count"))
else:
    df = None
    print("RUN_PROFILE=view: skipping dataset load; saved CSV tables will be loaded below.")


## Run Or Reuse Q3 Benchmark

`run_full` runs MIMIC oversampling inside each training fold, evaluates the GBC/LR protocol, and saves row-level and summary CSVs. `view` skips fitting and uses filenames reconstructed from `DATASET` and the config.

In [ ]:
if config.should_run_experiment:
    run_q3_benchmark(df, config, progress=print_progress_event)
else:
    print("RUN_PROFILE=view: skipping experiment run.")

print("Q3 result table filenames:")
for table, path in q3_result_table_paths(config).items():
    print(f"{table}: {path}")
display(q3_result_table_manifest(config))


## Load Saved Result Tables

Performance summaries and plots below read from the CSV tables saved by the benchmark.

In [ ]:
fold_results, q3_summary, manuscript_table = load_q3_result_tables(config)

display(q3_summary)
display(fold_results.head())
display(manuscript_table)


## Plot Metric Comparison

In [ ]:
fig, ax = plot_q3_metric_comparison(manuscript_table, classifier="GBC", metric="AUC")
